# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/umaimakhalid17/ML/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

Two findings from `docs/flyrank-seo-research-march-2026.pdf`, read constructively — not to score
a point, but to ask whether the validation design carries the claim.

### Finding #4 — "The Freshness Multiplier" (tagged CONFIRMED)

The paper's headline number: *"365+ day content that was refreshed within 30 days shows 3.2x
health boost (10.7 → 34.5) and 57x more impressions (71 → 4039)."* It calls refresh timing "one
of the strongest measured levers available."

**Where does the label/comparison come from?** A before/after comparison on pages that *were*
refreshed — not a controlled comparison against similar pages that weren't. The paper is careful
elsewhere (it flags its own 361+ freshness bucket as unstable: "283 growing pages versus only 1
declining"), but the refresh-timing headline doesn't get the same caveat, even though it's built
the same way: someone chose which 365+ day pages to refresh, and "already had upside" is a
plausible reason a page gets chosen for a refresh in the first place.

**Does the validation design carry the claim?** Not for a causal read. This is a **selection bias**
question: without knowing whether refreshed pages were chosen *because* an editor already saw
recovery potential (existing backlinks, seasonal demand returning, a competitor's page also
slipping), the 57x impression jump could be (part) the refresh, (part) the selection. The paper's
own action item — "run a recurring refresh program" — is a reasonable recommendation, but the
honest framing is **decision-support** ("pages like these are worth prioritizing for refresh"),
not **causal** ("refreshing produces 57x impressions"), and the paper's stronger language edges
past that line.

### Finding #1 — "The Anatomy of Growing Content" (tagged CONFIRMED)

The claim: growing pages are 37.6% longer and 20% younger than declining pages, at real scale
(74.8K rising vs 45.6K falling).

**Where does the label come from?** `up`/`down` direction is a 30-day-vs-prior-30-day impression
comparison — the exact same *current-window proxy* the flyrank-data skill warns about for
`trend_direction` in the starter CSV I'm using. It's a snapshot classification, not an observed
future outcome; a page currently in the `up` bucket isn't guaranteed to still be growing next
month, and the comparison says nothing about whether growth *caused* the length/age difference
or whether more effort was already being put into younger, promising pages before they grew.

**Does the validation design carry the claim?** The sample size argument is genuinely strong here
(n in the tens of thousands, this isn't a tiny-bucket problem) — but it's still a **cross-sectional
association**, not a design that supports the paper's own recommendation to "expand thin pages
that already earn impressions" as a growth lever. That recommendation implicitly claims word
count causes growth; the evidence shown only supports "growing and longer co-occur." A matched or
before/after design on the *same* pages over time would be needed to make the causal version of
this claim.

In [1]:
# No query needed here -- this section is a critical read of a public PDF, not a new
# computation. The one thing worth verifying is that the paper itself already flags instability
# in a comparably-sized bucket, which is the basis for my Finding #4 critique above.
print("Finding #4's own caveat (quoted structure, paraphrased): the 361+ freshness bucket is")
print("flagged unstable at n=284 (283 growing vs 1 declining) -- the same selection-bias logic")
print("applies to the refresh-timing headline, which the paper does not flag the same way.")


Finding #4's own caveat (quoted structure, paraphrased): the 361+ freshness bucket is
flagged unstable at n=284 (283 growing vs 1 declining) -- the same selection-bias logic
applies to the refresh-timing headline, which the paper does not flag the same way.


## 2. My model under an honest split (before/after)

Re-running `w05`'s logistic regression under a **naive random split** ("before") next to the
**client-grouped split** ("after") on the identical feature set, to see how much of the random
split's apparent skill was memorizing client identity.

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score

np.random.seed(42)
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

numeric_features = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d",
    "days_with_impressions", "days_with_sessions",
    "content_age_days", "days_since_last_update",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
]
categorical_features = [
    "competition_level", "content_type", "main_intent",
    "age_tier", "freshness_tier", "word_count_tier", "impression_tier", "position_tier",
]
missingness_flag_cols = ["search_volume", "competition", "cpc", "word_count", "char_count"]

X = df[numeric_features + categorical_features].copy()
for c in missingness_flag_cols:
    X[f"has_{c}"] = df[c].notna().astype(int)
X[numeric_features] = X[numeric_features].fillna(0)
X[categorical_features] = X[categorical_features].fillna("unknown")
num_cols_final = numeric_features + [f"has_{c}" for c in missingness_flag_cols]

y = df["is_declining_label"].values
groups = df["client_id"].values

preprocess = ColumnTransformer([
    ("num", StandardScaler(), num_cols_final),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
])

# BEFORE -- naive random split (rows from the same client can land on both sides)
Xtr_r, Xte_r, ytr_r, yte_r = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
lr_random = Pipeline([("pre", preprocess), ("clf", LogisticRegression(max_iter=3000, random_state=42))])
lr_random.fit(Xtr_r, ytr_r)
auc_random = roc_auc_score(yte_r, lr_random.predict_proba(Xte_r)[:, 1])

# AFTER -- client-grouped split (same one w05 uses)
gss = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))
Xtr_g, Xte_g = X.iloc[train_idx], X.iloc[test_idx]
ytr_g, yte_g = y[train_idx], y[test_idx]
lr_grouped = Pipeline([("pre", preprocess), ("clf", LogisticRegression(max_iter=3000, random_state=42))])
lr_grouped.fit(Xtr_g, ytr_g)
auc_grouped = roc_auc_score(yte_g, lr_grouped.predict_proba(Xte_g)[:, 1])

print(f"BEFORE -- naive random split AUC:   {auc_random:.3f}")
print(f"AFTER  -- client-grouped split AUC:  {auc_grouped:.3f}")
print(f"Gap:                                 {auc_random - auc_grouped:.3f}")


BEFORE -- naive random split AUC:   0.694
AFTER  -- client-grouped split AUC:  0.604
Gap:                                 0.090


**The gap is the finding.** The naive random split reports AUC 0.694; the honest, client-grouped
split reports 0.604 — a real 0.09-point gap. That's memorization, not skill: with a random split,
rows from the same client end up on both sides of train/test, so the model partly learns
"this client's pages behave like X" rather than a pattern that transfers to a client it has never
seen. 0.604 — not 0.694 — is the number I report everywhere else in this capstone (`w05`, `w07`,
the paper).

## 3. Leakage audit

The same hunt from `w03` (feature-leakage-check), re-run on this lane's final feature set —
including the "add the leaky column back and watch the score" test that proves the test harness
itself is sensitive to leakage, not just quiet.

In [3]:
# Attack checklist, item by item, on the exact features w05/w06 use:
feature_set = set(num_cols_final) | set(categorical_features)
label_derived = {"trend_direction", "trend_pct", "is_declining_label"}
product_flags = {"health_score", "priority_score", "action_type", "refresh_tier"}  # not in this dataset at all

print("[1] Label-derived columns in the feature set (must be empty):", feature_set & label_derived)
print("[2] Product-flag columns in the feature set (must be empty; these columns don't exist here):",
      feature_set & product_flags)
print("[3] Population check: rows are kept unconditionally (impressions_90d > 0 by construction of")
print("    this slice) -- no filter depends on anything from the outcome window.")
print("[4] Split grouped by client_id: confirmed in Section 2 (0 client overlap between folds).")
print("[5] Base rate printed next to every metric: done in w05 Section 3 and this notebook's Section 2.")
print()

# The confession test: deliberately ADD trend_pct (the column the label is thresholded from)
# and watch the score jump toward 1.0 -- if it doesn't, the test harness itself is broken.
X_leaky = X.copy()
X_leaky["trend_pct_LEAKY"] = df["trend_pct"].fillna(0)
num_cols_leaky = num_cols_final + ["trend_pct_LEAKY"]
preprocess_leaky = ColumnTransformer([
    ("num", StandardScaler(), num_cols_leaky),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
])
Xtr_l, Xte_l = X_leaky.iloc[train_idx], X_leaky.iloc[test_idx]
lr_leaky = Pipeline([("pre", preprocess_leaky), ("clf", LogisticRegression(max_iter=3000, random_state=42))])
lr_leaky.fit(Xtr_l, ytr_g)
auc_leaky = roc_auc_score(yte_g, lr_leaky.predict_proba(Xte_l)[:, 1])

print(f"HONEST grouped-split AUC (no trend_pct):      {auc_grouped:.3f}")
print(f"LEAKY grouped-split AUC (trend_pct injected):  {auc_leaky:.3f}")
print("Confession confirmed: adding the column the label is thresholded FROM collapses the")
print("problem from 'predict decline' to 'read the answer key' -- trend_pct/trend_direction")
print("stay excluded everywhere else in this repo.")


[1] Label-derived columns in the feature set (must be empty): set()
[2] Product-flag columns in the feature set (must be empty; these columns don't exist here): set()
[3] Population check: rows are kept unconditionally (impressions_90d > 0 by construction of
    this slice) -- no filter depends on anything from the outcome window.
[4] Split grouped by client_id: confirmed in Section 2 (0 client overlap between folds).
[5] Base rate printed next to every metric: done in w05 Section 3 and this notebook's Section 2.



HONEST grouped-split AUC (no trend_pct):      0.604
LEAKY grouped-split AUC (trend_pct injected):  0.998
Confession confirmed: adding the column the label is thresholded FROM collapses the
problem from 'predict decline' to 'read the answer key' -- trend_pct/trend_direction
stay excluded everywhere else in this repo.


## 4. Claim rewrite

**My boldest sentence, as first drafted (`w04`, Signal 2):**

> "CTR clearly declines as position worsens, both ways of measuring it."

**Rewritten in safe, claim-ladder language:**

> "In this 30,000-row starter slice, mean click-through rate **was observed to fall** at every
> step from `top_3` (2.76%) through `deep` (0.15%) positions, and this pattern **held** whether
> measured as the average of each page's own CTR or as the impression-weighted portfolio rate.
> This is a **directional, decision-support** signal — pages with CTR well below their position
> tier's typical range are **associated with** higher observed decline rates in this slice
> (Section 2/3, `w04`) — not a claim about *why* CTR falls, or about what any individual page's
> CTR will do next."

The first version reads as a settled fact about how search results work in general. The rewrite
scopes it to *this dataset, this period*, keeps the "associated with" language instead of implying
mechanism, and doesn't promise anything about a specific page's future — which matches exactly
what a cross-sectional, non-experimental comparison can support.

In [4]:
# No new computation needed -- Section 4 is a rewrite exercise against evidence already
# produced in w04. Restating the underlying numbers here so the rewrite is checkable in one place.
print("Underlying evidence for the claim above (from w04 Signal 2):")
print("top_3: 2.76% (n=1,116) -> page_1: 0.65% (n=11,814) -> striking: 0.32% (n=7,304)")
print("-> page_3_5: 0.22% (n=7,242) -> deep: 0.15% (n=1,319)")
print("All n well above the ~50-row sample floor; monotonic in both the row-mean and")
print("impression-weighted views.")


Underlying evidence for the claim above (from w04 Signal 2):
top_3: 2.76% (n=1,116) -> page_1: 0.65% (n=11,814) -> striking: 0.32% (n=7,304)
-> page_3_5: 0.22% (n=7,242) -> deep: 0.15% (n=1,319)
All n well above the ~50-row sample floor; monotonic in both the row-mean and
impression-weighted views.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.